In [1]:
import os

In [2]:
os.getcwd()

'c:\\Users\\harsh\\Desktop\\text-summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
os.getcwd()

'c:\\Users\\harsh\\Desktop\\text-summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml,creat_directories

In [ ]:
class ConfigurationManager:
    def __init__(self,
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH):

        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        creat_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion
        creat_directories([config.root_dir])

        data_ingestion_config= DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [22]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size
import requests


In [53]:
class DataIngestion:
    def __init__(self,config: DataIngestionConfig):
        self.config=config

    def download_file(self):

        print("URL:", self.config.source_url)
        print("PATH:", self.config.local_data_file)

        response = requests.get(self.config.source_url)

        print("STATUS:", response.status_code)
        print("CONTENT TYPE:", response.headers.get("Content-Type"))
        print("DOWNLOADED:", len(response.content), "bytes")

        response.raise_for_status()

        with open(self.config.local_data_file, "wb") as f:
            f.write(response.content)

        print(
            "SAVED SIZE:",
            os.path.getsize(self.config.local_data_file),
            "bytes"
        )

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        zip_path = self.config.local_data_file

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            for file in zip_ref.namelist():
                try:
                    print("Testing:", file)
                    zip_ref.read(file)
                except Exception as e:
                    print("PROBLEM:", file)
                    print(type(e).__name__, e)
                    break

In [33]:
try:
    config= ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    # data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-06 23:11:36,523: INFO: common: yaml file: config\config.yaml loaded succesfully:]
[2026-10-06 23:11:36,529: INFO: common: yaml file: params.yaml loaded succesfully:]
[2026-10-06 23:11:36,534: INFO: common: create director at : artifacts:]
[2026-10-06 23:11:36,539: INFO: common: create director at : artifacts/data_ingestion:]
URL: https://github.com/entbappy/Branching-tutorial/raw/refs/heads/master/samsumdata.zip
PATH: artifacts/data_ingestion/data.zip
STATUS: 200
CONTENT TYPE: application/zip
DOWNLOADED: 23627009 bytes
SAVED SIZE: 23627009 bytes


In [31]:
import requests
import os
import zipfile

url = "https://github.com/entbappy/Branching-tutorial/raw/refs/heads/master/samsumdata.zip"
path = "artifacts/data_ingestion/data.zip"

response = requests.get(url)
response.raise_for_status()

print("Downloaded:", len(response.content))

with open(path, "wb") as f:
    f.write(response.content)

print("After save:", os.path.getsize(path))
print("Is ZIP:", zipfile.is_zipfile(path))

Downloaded: 23627009
After save: 23627009
Is ZIP: True


In [35]:
import os
import zipfile

path = "artifacts/data_ingestion/data.zip"

print("Size:", os.path.getsize(path), "bytes")
print("Valid ZIP:", zipfile.is_zipfile(path))

with open(path, "rb") as f:
    print(f.read(100))

Size: 23627009 bytes
Valid ZIP: True
b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x0cF\xafVb\xee\xb8\xcc\xde\x82x\x00j\x99x\x00\x08\x00\x00\x00data.zip\x00\x10,\xef\xd3PK\x03\x04\x14\x00\x00\x00\x08\x00\xa8a\xa9V\xf1\x96\xb0\x0b!\x02\x03\x00\xf5\x16\x08\x00\x0f\x00\x00\x00samsum-test.csv\xac]\xcb\x8e\x1b]\x11^\x13\x89w\xe8'
